In [6]:
import re
from pathlib import Path
import rioxarray

# --- Setup paths ---
basepath = Path("D:/MyDrive/Stability/RawData")
region_map = {
        "EXWR": "Extra_Wainwright",
        "UTNQ": "UTQ_NSQ",
        "WRUT": "Wainwright_UTQ",
        "PBKT": "Prudhoe_Bay_Katovik",
        "NQKA": "NSQ-KAK",
        "CODT": "Colville_Delta",
        "KAMK": "Kaktovik_MK_Delta",
    }
outdir = basepath / "Monthly_Averages" / "Beaufort_month_year"
outdir.mkdir(parents=True, exist_ok=True)

# Regex for YYYYMMDD
date_regex = re.compile(r'(\d{4})(\d{2})(\d{2})')

# --- Loop regions ---
for code, subdir in region_map.items():
    print(f"Processing region {subdir}")
    strain_folder = basepath / subdir / "Strain_Files_2025"
    if not strain_folder.exists():
        print(f"Skipping missing folder: {strain_folder}")
        continue

    # Dicts keyed by (year, month)
    monthly_sum = {}
    monthly_count = {}
    monthly_ref = {}

    tifs = list(strain_folder.glob("*.tif"))
    print(f"Found {len(tifs)} files in {subdir}")

    for tif in tifs:
        fname = tif.name
        match = date_regex.search(fname)
        if not match:
            print(f"Skipping {tif}, cannot parse date")
            continue

        year = int(match.group(1))
        month = int(match.group(2))
        key = (year, month)

        try:
            ds = rioxarray.open_rasterio(
                tif, masked=True, chunks={'x': 500, 'y': 500}
            ).squeeze()

            valid_mask = ~ds.isnull()
            if key not in monthly_sum:
                monthly_sum[key] = ds.fillna(0).copy()
                monthly_count[key] = valid_mask.astype(int).copy()
                monthly_ref[key] = ds
            else:
                ds_aligned = ds.rio.reproject_match(monthly_sum[key])
                valid_mask_aligned = ~ds_aligned.isnull()
                monthly_sum[key].data += ds_aligned.fillna(0).data
                monthly_count[key].data += valid_mask_aligned.astype(int).data

        except Exception as e:
            print(f"Could not read {tif}: {e}")

    # --- Compute & save monthly means per year ---
    for (year, month), sum_arr in monthly_sum.items():
        count_arr = monthly_count[(year, month)]
        mean = sum_arr / count_arr
        mean = mean.where(count_arr > 1)

        mean_file = outdir / f"{subdir}_Monthly_Mean_{year}_{month:02d}.tif"
        count_file = outdir / f"{subdir}_Monthly_Count_{year}_{month:02d}.tif"

        mean.rio.to_raster(mean_file)
        count_arr.rio.to_raster(count_file)
        print(f"Saved {mean_file} and {count_file}")

Processing region Extra_Wainwright
Found 149 files in Extra_Wainwright
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Beaufort_month_year\Extra_Wainwright_Monthly_Mean_2017_05.tif and D:\MyDrive\Stability\RawData\Monthly_Averages\Beaufort_month_year\Extra_Wainwright_Monthly_Count_2017_05.tif
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Beaufort_month_year\Extra_Wainwright_Monthly_Mean_2017_06.tif and D:\MyDrive\Stability\RawData\Monthly_Averages\Beaufort_month_year\Extra_Wainwright_Monthly_Count_2017_06.tif
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Beaufort_month_year\Extra_Wainwright_Monthly_Mean_2017_07.tif and D:\MyDrive\Stability\RawData\Monthly_Averages\Beaufort_month_year\Extra_Wainwright_Monthly_Count_2017_07.tif
Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Beaufort_month_year\Extra_Wainwright_Monthly_Mean_2017_08.tif and D:\MyDrive\Stability\RawData\Monthly_Averages\Beaufort_month_year\Extra_Wainwright_Monthly_Count_2017_08.tif
Saved D:\MyDrive\Stab